# 1. From signals to portfolio returns

The two signals are Dynamic Mode Decomposition (DMD) and betting against inflation beta. This notebook shows how to make the signals, fit the combination rules, and construct and test the portfolios. The portfolio comparison covers 2010–2024. A signal date is the date when the score is known. The trade occurs at the next close. The holding return ends at the following close.

The analysis notebooks use the saved output of this test. You can run all their cells from the supplied files. The full portfolio replay also needs the licensed sf-quant score files specified below. Source-level score generation needs the stock data and access to the risk service.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

# Open this notebook from the submission folder.
DATA = Path('dataset')
assert (DATA / 'analysis_panel.parquet').is_file()
plt.rcParams.update({'figure.figsize': (11, 5), 'axes.spines.top': False,
                     'axes.spines.right': False})


## 1.1 Make the DMD stock score

Dynamic mode decomposition (DMD) uses recent stock prices to find patterns. The code makes a price direction forecast for each stock. It selects a price window from results that are known on the signal date. The forecast is for the close after the trade to the next close. This date order prevents the use of future results.

In [2]:
import numpy as np

def dmd_predicts_up(
    prices: np.ndarray, horizon: int = 1, rank: int = 25, start_horizon: int = 0,
) -> bool:
    """Forecast direction between two future offsets from the last close."""
    prices = np.asarray(prices, dtype=float)
    n = len(prices)
    if (n < 4 or start_horizon < 0 or horizon <= start_horizon
            or not np.isfinite(prices).all() or (prices <= 0).any()):
        return False
    # Use the dated price values. This is the model input.
    logp = prices.copy()
    row_len = n // 2 + 1
    hankel = np.column_stack([logp[i:i + row_len] for i in range(n - row_len + 1)])
    x, y = hankel[:, :-1], hankel[:, 1:]
    try:
        u, singular, vh = np.linalg.svd(x, full_matrices=False)
        r = min(rank, len(singular))
        if r == 0 or singular[r - 1] <= np.finfo(float).eps:
            return False
        ur, vr = u[:, :r], vh[:r, :].T
        inv_s = np.diag(1.0 / singular[:r])
        reduced = ur.T @ y @ vr @ inv_s
        eigenvalues, modes = np.linalg.eig(reduced)
        good = np.isfinite(eigenvalues) & (np.abs(eigenvalues) > 1e-12)
        if not good.any():
            return False
        eigenvalues, modes = eigenvalues[good], modes[:, good]
        phi = y @ vr @ inv_s @ modes
        amplitudes = np.linalg.lstsq(phi, hankel, rcond=None)[0][:, 0]
        last = phi[0] @ (amplitudes * eigenvalues ** (hankel.shape[1] - 1 + start_horizon))
        future = phi[0] @ (amplitudes * eigenvalues ** (hankel.shape[1] - 1 + horizon))
        return bool(np.isfinite(last) and np.isfinite(future) and future > last)
    except (np.linalg.LinAlgError, FloatingPointError):
        return False


def walk_forward_stock_signal(
    prices: np.ndarray,
    *,
    windows: tuple[int, ...] = tuple(range(3, 25)),
    selection_lookback: int = 60,
    invert: bool = True,
) -> np.ndarray:
    """Daily online hit-rate selection for each stock.

    The forecast covers the close after execution (t+1) to the next close
    (t+2). Window selection uses only those outcomes known by signal date t.
    Output is 0 until enough history exists, otherwise
    +1/-1; the model's ``INVERT_SIGNALS=True`` convention is retained.
    """
    p = np.asarray(prices, dtype=float)
    t_count = len(p)
    pred = np.full((len(windows), t_count), np.nan)
    for wi, size in enumerate(windows):
        for t in range(size - 1, t_count):
            segment = p[t-size+1:t+1]
            if np.isfinite(segment).all():
                pred[wi, t] = 1.0 if dmd_predicts_up(
                    segment, horizon=2, start_horizon=1,
                ) else -1.0
    out = np.zeros(t_count, dtype=np.float32)
    for t in range(t_count):
        # At date t, only the result of a forecast made at t-2 is known.
        hi = t - 2
        lo = hi - selection_lookback + 1
        if lo < 0 or hi < lo:
            continue
        future = p[2:t+1] / p[1:t] - 1.0
        actual = np.sign(future)
        actual[~np.isfinite(future)] = 0
        scores = []
        for wi in range(len(windows)):
            hist = pred[wi, lo:hi+1]
            truth = actual[lo:hi+1]
            valid = np.isfinite(hist) & (truth != 0)
            scores.append(np.mean(np.sign(hist[valid]) == truth[valid]) if valid.sum() == selection_lookback else -1.0)
        best = int(np.argmax(scores))
        if scores[best] >= 0 and np.isfinite(pred[best, t]):
            out[t] = pred[best, t] * (-1 if invert else 1)
    return out


In [3]:
def make_dmd_scores(assets: pl.DataFrame, shard: int, shard_count: int) -> pl.DataFrame:
    """Make DMD scores for one group of stocks."""
    # Use the same trade dates for each stock.
    stock_ids = (assets.filter(pl.col('in_universe')).select('barrid').unique()
                 .sort('barrid')['barrid'].to_list()[shard::shard_count])
    source = assets.select('date', 'barrid', 'price', 'in_universe')
    calendar = source.select('date').unique().sort('date')
    rows = []
    for stock in source.filter(pl.col('barrid').is_in(stock_ids)).partition_by('barrid'):
        aligned = calendar.join(stock, on='date', how='left', validate='1:1').sort('date')
        # The model uses prices that are known on the signal date.
        scores = walk_forward_stock_signal(aligned['price'].to_numpy())
        rows.append(aligned.with_columns(pl.Series('dmd', scores)).filter(
            pl.col('in_universe') & (pl.col('date') >= __import__('datetime').date(2005, 1, 1))
        ).select(pl.col('date').alias('signal_date'), 'barrid', 'dmd'))
    return pl.concat(rows)


## 1.2 Make the inflation beta score

The code uses the CPI value that was available at each month end. It turns daily stock returns into monthly returns. It then measures how the stock return moves with the CPI change. The score is the negative of this value. Thus, a stock with more negative inflation sensitivity gets a higher score.

In [4]:
import polars as pl
def vintage_inflation_changes(vintages: pl.DataFrame) -> pl.DataFrame:
    """Calculate the last available month-over-month change in YoY CPI per vintage."""
    required = {"vintage_date", "observation_date", "cpi"}
    if missing := required - set(vintages.columns):
        raise ValueError(f"Missing CPI vintage fields: {sorted(missing)}")
    vintage_rows = (
        vintages.sort(["vintage_date", "observation_date"])
        .with_columns(
            ((pl.col("cpi") / pl.col("cpi").shift(12).over("vintage_date") - 1) * 100)
            .alias("inflation_yoy")
        )
        .with_columns(
            (pl.col("inflation_yoy") - pl.col("inflation_yoy").shift(1).over("vintage_date"))
            .alias("inflation_change")
        )
        .drop_nulls("inflation_change")
        .group_by("vintage_date", maintain_order=True)
        .last()
        .select(pl.col("vintage_date").alias("date"), "inflation_change")
        .sort("date")
    )
    return vintage_rows


def make_inflation_scores(assets: pl.DataFrame, vintages: pl.DataFrame) -> pl.DataFrame:
    """Make a dated stock score from inflation sensitivity."""
    # Use the CPI value that was available at each month end.
    cpi = vintage_inflation_changes(vintages)
    # Stock daily returns are percentage points. Convert them to decimals.
    monthly = (
        assets.lazy().select(pl.col('date').dt.month_end(), 'barrid', 'return')
        .group_by('date', 'barrid')
        .agg(((1 + pl.col('return') / 100).product() - 1).alias('stock_return'),
             pl.col('return').count().alias('n_returns'))
        .filter(pl.col('n_returns') > 0)
        .join(cpi.lazy(), on='date').sort('date').collect()
    )
    # Use up to 60 months. Require at least 36 valid months.
    result = (
        monthly.with_columns((pl.col('stock_return') * pl.col('inflation_change')).alias('xy'),
                             pl.col('inflation_change').pow(2).alias('yy'))
        .rolling('date', period='60mo', group_by='barrid')
        .agg(pl.len().alias('n'), pl.col('stock_return').mean().alias('x'),
             pl.col('inflation_change').mean().alias('y'),
             pl.col('xy').mean(), pl.col('yy').mean())
        .with_columns((pl.col('yy') - pl.col('y').pow(2)).alias('variance'))
        .filter((pl.col('n') >= 36) & (pl.col('variance') > 0))
        .select(pl.col('date').alias('available_date'), 'barrid',
                (-(pl.col('xy') - pl.col('x') * pl.col('y')) / pl.col('variance')).alias('inflation_beta'))
    )
    return result

def join_dated_scores(dmd_shards: list[pl.DataFrame], inflation: pl.DataFrame) -> pl.DataFrame:
    """Join each stock score to the date when it is known."""
    return (pl.concat(dmd_shards).sort('signal_date')
            .join_asof(inflation.sort('available_date'), left_on='signal_date',
                       right_on='available_date', by='barrid', strategy='backward', tolerance='45d')
            .select('signal_date', 'barrid', 'dmd', 'inflation_beta'))


### 1.2.1 Make the dated CPI source file

The Federal Reserve ALFRED service gives the CPI value that was available at each month end. The next code cell shows how to save those dated values. Keep that file if you need an exact later run.

In [5]:
import calendar
import csv
import io
from datetime import date
from urllib.request import urlopen
ALFRED_URL = 'https://alfred.stlouisfed.org/graph/alfredgraph.csv'
def fetch_alfred_month_end_vintages(start: date, end: date) -> pl.DataFrame:
    """Download one point-in-time CPIAUCSL snapshot per month from ALFRED.

    ``vintage_date`` is month end, not the observation month. Consumers must
    join the latest vintage whose date is on or before the signal date.
    """
    rows = []
    year, month = start.year, start.month
    while (year, month) <= (end.year, end.month):
        vintage = date(year, month, calendar.monthrange(year, month)[1])
        url = f"{ALFRED_URL}?id=CPIAUCSL&vintage_date={vintage.isoformat()}"
        with urlopen(url, timeout=30) as response:
            csv_text = response.read().decode("utf-8")
        for row in csv.DictReader(io.StringIO(csv_text)):
            value = row[f"CPIAUCSL_{vintage:%Y%m%d}"]
            if value and value != ".":
                if date.fromisoformat(row["observation_date"]) >= vintage.replace(day=1):
                    raise ValueError(f"Unexpected observation in vintage {vintage}: {row}")
                rows.append({
                    "vintage_date": vintage,
                    "observation_date": date.fromisoformat(row["observation_date"]),
                    "cpi": float(value),
                })
        if month == 12:
            year, month = year + 1, 1
        else:
            month += 1
    return pl.DataFrame(rows).sort(["vintage_date", "observation_date"])


# With network access, make the dated CPI source file once.
# cpi = fetch_alfred_month_end_vintages(date(2000, 1, 1), date(2025, 12, 31))
# cpi.write_parquet(Path('/path/to/licensed/source/cpi_alfred_vintages.parquet'))


## 1.3 Make the dated input files

Join each DMD score to the latest inflation score known at that time. The panel code aligns the signal date, trade date, and holding return. It also saves the benchmark weights. The risk files save daily stock risk in the same stock order as the panel. These files are in `dataset/`.

In [6]:
import polars as pl
def build_panel(assets: pl.DataFrame, forecasts: pl.DataFrame, exits: pl.DataFrame | None = None) -> pl.DataFrame:
    """Align dates on a common exchange calendar, including for stocks that exit."""
    calendar = assets.select("date").unique().sort("date").with_columns(
        pl.col("date").shift(1).alias("signal_date"),
        pl.col("date").shift(-1).alias("label_available_date"),
    )
    members = assets.filter(pl.col("in_universe")).with_columns(
        (pl.col("market_cap") / pl.col("market_cap").sum().over("date")).alias("benchmark_weight")
    )
    # sf-quant stores daily returns in percentage points. CRSP fallbacks are decimals.
    realized = assets.select(
        pl.col("date").alias("label_available_date"),
        "barrid",
        (pl.col("return") / 100).alias("holding_return"),
    )
    if exits is not None:
        realized = realized.join(
            exits.select("label_available_date", "barrid", "delret", "crsp_return"),
            on=["label_available_date", "barrid"], how="full", coalesce=True, validate="1:1",
        ).with_columns(
            pl.coalesce("holding_return", "delret", "crsp_return").alias("holding_return")
        ).drop("delret", "crsp_return")
    panel = (
        members.join(calendar, on="date", validate="m:1")
        .drop_nulls(["signal_date", "label_available_date"])
        .join(forecasts, on=["signal_date", "barrid"], how="left", validate="m:1")
        .join(realized, on=["label_available_date", "barrid"], how="left", validate="m:1")
        .with_columns(
            pl.col("holding_return").is_null().sum().over("date").alias("carried_names"),
            pl.col("holding_return").fill_null(0),
        )
        .with_columns(
            (pl.col("benchmark_weight") * pl.col("holding_return")).sum().over("date").alias("benchmark_return")
        )
        .with_columns(
            (pl.col("holding_return") - pl.col("predicted_beta") * pl.col("benchmark_return"))
            .alias("forward_residual_return")
        )
    )
    return panel.sort(["date", "barrid"])


In [7]:
def save_risk_year(panel: pl.DataFrame, risk_source, year: int, destination: Path) -> None:
    """Save the daily risk data for one calendar year."""
    arrays = {}
    days = panel.filter(pl.col('date').dt.year() == year).partition_by('date', maintain_order=True)
    for day in days:
        key = str(day['date'][0])
        ids = day['barrid'].to_list()
        # Save stock IDs and risk arrays in the same order.
        exposure, factor_risk, stock_risk = risk_source.construct_factor_model_components(day['date'][0], ids)
        arrays[key + '_ids'] = np.asarray(ids, dtype=str)
        arrays[key + '_E'] = exposure
        arrays[key + '_F'] = factor_risk
        arrays[key + '_S'] = stock_risk
    np.savez_compressed(destination, **arrays)


### 1.3.1 Add stock exit returns

A stock can leave the data after it is in a portfolio. The next code cell gets a return from the licensed CRSP files when the main stock file has no return. This step keeps the stock in the test for its last holding period.

In [8]:
def make_exit_audit(database: Path, destination: Path) -> None:
    """Find a return for a stock that leaves the stock data."""
    from datetime import date
    delist = pl.scan_parquet(database / 'production/crsp_delisting/crsp_delisting.parquet')
    rows = []
    for year in range(2005, 2026):
        paths = [database / 'production/assets' / f'assets_{y}.parquet'
                 for y in range(year, min(year + 1, 2025) + 1)]
        assets = pl.scan_parquet(paths)
        if year < 2025:
            assets = assets.filter(pl.col('date') <= date(year + 1, 1, 15))
        calendar = assets.select('date').unique().sort('date').with_columns(
            pl.col('date').shift(-1).alias('label_available_date'))
        # Select stocks with no return on the next trade date.
        missing = (
            assets.filter(pl.col('in_universe') & (pl.col('date').dt.year() == year))
            .select('date', 'barrid', pl.col('cusip').str.slice(0, 8).alias('cusip'))
            .join(calendar, on='date').drop_nulls('label_available_date')
            .join(assets.select(pl.col('date').alias('label_available_date'), 'barrid', 'return'),
                  on=['label_available_date', 'barrid'], how='left')
            .filter(pl.col('return').is_null()))
        crsp = pl.scan_parquet(database / 'research/crsp_daily' / f'crsp_daily_{year}.parquet')
        mapping = crsp.select('date', 'cusip', 'permno').drop_nulls().unique().group_by(
            'date', 'cusip').agg(pl.col('permno').first(), pl.len().alias('count')).filter(pl.col('count') == 1)
        return_paths = [database / 'research/crsp_daily' / f'crsp_daily_{y}.parquet'
                        for y in range(year, min(year + 1, 2025) + 1)]
        crsp_returns = pl.scan_parquet(return_paths).select(
            pl.col('date').alias('label_available_date'), 'permno',
            pl.when(pl.col('ret').is_finite() & (pl.col('ret') >= -1))
            .then(pl.col('ret')).alias('crsp_return'))
        matched = (
            missing.join(mapping, on=['date', 'cusip'], how='left')
            .join(delist.select('permno', pl.col('delistingdt').alias('label_available_date'), 'delret'),
                  on=['permno', 'label_available_date'], how='left', validate='m:1')
            .join(crsp_returns, on=['permno', 'label_available_date'], how='left', validate='m:1')
            .select('date', 'label_available_date', 'barrid', 'permno', 'delret', 'crsp_return').collect())
        rows.append(matched)
    pl.concat(rows).write_parquet(destination)

# Use the licensed CRSP files to make the source audit file.
# make_exit_audit(Path('/path/to/licensed/database'),
#                 Path('/path/to/licensed/source/exit_return_audit.parquet'))


In [9]:
def make_all_input_files(source_folder: Path, risk_source, output: Path) -> None:
    """Make the dated panel and all yearly risk files from source data."""
    output.mkdir(parents=True, exist_ok=True)
    end = date(2024, 12, 31)

    # Read the same stock fields for the full warm-up period.
    stock_files = [source_folder / f'assets_{year}.parquet' for year in range(1995, 2026)]
    assets = pl.scan_parquet(stock_files).select(
        'date', 'barrid', 'in_universe', 'market_cap', 'return',
        'predicted_beta', 'price').collect()
    vintages = pl.read_parquet(source_folder / 'cpi_alfred_vintages.parquet')
    exits = pl.read_parquet(source_folder / 'exit_return_audit.parquet')

    # Divide stocks into groups. Each group can run on a separate worker.
    # Keep the stock history and the full exchange calendar in each group.
    from concurrent.futures import ThreadPoolExecutor
    with ThreadPoolExecutor(max_workers=8) as pool:
        dmd_groups = list(pool.map(lambda i: make_dmd_scores(assets, i, 128), range(128)))
    inflation = make_inflation_scores(assets, vintages)
    forecasts = join_dated_scores(dmd_groups, inflation)

    # The portfolio test starts in 2010. Keep 2005 to 2009 for training.
    test_assets = assets.filter(pl.col('date').is_between(date(2005, 1, 1), end))
    raw_panel = build_panel(test_assets, forecasts, exits)

    # Add the two Barra style scores known on each signal date.
    parts = []
    years = raw_panel.with_columns(pl.col('signal_date').dt.year().alias('_year'))
    for year in years.partition_by('_year', include_key=False):
        styles = risk_source.load_exposures(
            year['signal_date'].min(), year['signal_date'].max(), False,
            ['USSLOWL_MOMENTUM', 'USSLOWL_LTREVRSL'])
        parts.append(year.join(styles.rename({'date': 'signal_date'}),
                               on=['signal_date', 'barrid'], how='left', validate='m:1'))
    panel = pl.concat(parts).sort('date', 'barrid')
    panel.write_parquet(output / 'analysis_panel.parquet')

    # The risk arrays must have the same stock order as the dated panel.
    for year in range(2010, 2025):
        save_risk_year(panel, risk_source, year, output / f'risk_{year}.npz')

# With licensed source files and sf-quant access, use:
# import sf_quant.data as risk_source
# risk_source.env(root=str(Path.home()), database='production')
# make_all_input_files(Path('/path/to/licensed/source'), risk_source, Path('rebuilt_inputs'))


In [10]:
# Check the supplied input files before the portfolio test.
import pyarrow.parquet as pq
panel_file = pq.ParquetFile(DATA / 'analysis_panel.parquet')
need = {'date', 'signal_date', 'label_available_date', 'barrid', 'dmd',
        'inflation_beta', 'holding_return', 'benchmark_weight',
        'USSLOWL_MOMENTUM', 'USSLOWL_LTREVRSL'}
assert need.issubset(panel_file.schema.names)
assert len(list(DATA.glob('risk_20*.npz'))) == 15
print(f"Stock rows: {panel_file.metadata.num_rows:,}")
print('Risk years: 15')


Stock rows: 14,939,812
Risk years: 15


## 1.4 Make the adjusted DMD score

For each signal date, fit the original DMD scores to the sf-quant momentum and reversal scores across stocks. Subtract the fitted score to get the adjusted DMD score. When a stock lacks one of the two style scores, leave the adjusted score missing. The adjusted ridge fit uses complete rows. At prediction time, a missing adjusted score contributes zero to the forecast, and the inflation score can still contribute. The fit uses the same signal date as the DMD score.

Use `scores_2004.parquet` through `scores_2024.parquet` as applicable to the signal dates in the panel. Each annual file needs `date`, `barrid`, `signal_name`, and `score`. The required signal names are `momentum` and `reversal`.

In [11]:
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
import os
from scipy.special import softmax

RULES = {'ridge_63': (63, 18), 'ridge_252': (252, 5),
         'ridge_1260': (1260, 2.5), 'equal_weight': (126, 1)}
BMA = {'dmd': 'bma_original', 'adjusted': 'bma_adjusted'}
SIGNALS = {signal: (signal, 'inflation_beta') for signal in BMA}
VARIANTS = {'long_short': False, 'long_only': True}
FIELDS = ['date', 'signal_date', 'label_available_date', 'barrid', 'dmd',
          'inflation_beta', 'forward_residual_return', 'holding_return',
          'predicted_beta', 'benchmark_weight']


def load_year(panel, year):
    return (pl.scan_parquet(panel).filter(pl.col('date').dt.year() == year)
            .select(FIELDS).sort('date', 'barrid').collect())


def prepare_year(panel, output, year, score_dir):
    data = load_year(panel, year)
    start, end = data['signal_date'].min(), data['signal_date'].max()
    paths = [score_dir / f'scores_{y}.parquet' for y in range(start.year, end.year + 1)]
    styles = (pl.scan_parquet(paths)
              .filter(pl.col('date').is_between(start, end),
                      pl.col('signal_name').is_in(['momentum', 'reversal']))
              .select('date', 'barrid', 'signal_name', 'score').collect())
    styles = styles.pivot(on='signal_name', index=['date', 'barrid'], values='score')
    joined = data.join(styles.rename({'date': 'signal_date'}),
                       on=['signal_date', 'barrid'], how='left', validate='m:1').sort('date', 'barrid')
    days = []
    coverage = []
    for day in joined.partition_by('date', maintain_order=True):
        raw = day['dmd'].to_numpy()
        style = day.select('momentum', 'reversal').to_numpy()
        matched = np.isfinite(raw) & np.isfinite(style).all(axis=1)
        if matched.sum() < 30:
            raise ValueError(f"Insufficient regular style scores on {day['date'][0]}: {matched.sum()}")
        x = style[matched] - style[matched].mean(axis=0)
        y = raw[matched] - raw[matched].mean()
        residual = np.full(len(raw), np.nan)
        residual[matched] = y - x @ np.linalg.lstsq(x, y, rcond=None)[0]
        days.append(day.select(FIELDS).with_columns(pl.Series('adjusted', residual)))
        coverage.append({'date': day['date'][0], 'matched_share': matched.mean()})
    data = pl.concat(days)
    pl.DataFrame(coverage).write_parquet(output / f'coverage_{year}.parquet')
    for signal, columns in SIGNALS.items():
        daily, _, _ = daily_ridge_inputs(data, signals=columns)
        daily.write_parquet(output / f'moments_{signal}_{year}.parquet')
    if year >= 2010:
        data.write_parquet(output / f'panel_{year}.parquet')




### 1.4.1 Measure DMD score and payoff overlap

This separate calculation uses two Barra styles and two sf-quant scores. It measures two things. The first is the share of DMD score variation that the styles fit. The second is the share of DMD's stock-level payoff associated with that fitted part. The payoff uses each stock's later market-beta-adjusted return. It is not a portfolio return. It is not proof that a style caused the payoff. The saved daily file and summary file are in `dataset/`.

In [12]:
# These four names identify two Barra styles and two sf-quant scores.
CONTROLS = {
    'barra_momentum': 'USSLOWL_MOMENTUM',
    'barra_reversal': 'USSLOWL_LTREVRSL',
    'sf_momentum': 'momentum',
    'sf_reversal': 'reversal'}
SPECS = {
    **{name: (name,) for name in CONTROLS},
    'momentum': ('barra_momentum', 'sf_momentum'),
    'reversal': ('barra_reversal', 'sf_reversal'),
    'barra': ('barra_momentum', 'barra_reversal'),
    'sf_quant': ('sf_momentum', 'sf_reversal'),
    'all': tuple(CONTROLS)}
def join_style_controls(panel):
    """Join same-signal-date style controls for a single calendar year."""
    import sf_quant.data as sfd

    start, end = panel["signal_date"].min(), panel["signal_date"].max()
    exposures = sfd.load_exposures(start, end, False, [CONTROLS["barra_momentum"], CONTROLS["barra_reversal"]])
    signals = sfd.load_scores(start, end, names=[CONTROLS["sf_momentum"], CONTROLS["sf_reversal"]])
    for source, names, value in (
        (exposures, ("barra_momentum", "barra_reversal"), None),
        (signals, ("sf_momentum", "sf_reversal"), "score"),
    ):
        if value:
            source = source.group_by("date", "barrid").agg(*(
                pl.col(value).filter(pl.col("signal_name") == CONTROLS[name]).first().alias(name)
                for name in names
            ))
        else:
            source = source.rename({CONTROLS[name]: name for name in names})
        panel = panel.join(source.rename({"date": "signal_date"}),
                             on=["signal_date", "barrid"], how="left", validate="m:1")
    return panel


def daily_attribution(dmd, y, styles):
    """OLS projection within one signal date. Payoff is exactly additive."""
    dmd = dmd - dmd.mean()
    y = y - y.mean()
    styles = styles - styles.mean(axis=0)
    ss = dmd @ dmd
    gross = np.abs(dmd).sum()
    if ss <= 0:
        return []
    result = []
    raw = (dmd @ y) / gross
    for spec, controls in SPECS.items():
        cols = [list(CONTROLS).index(c) for c in controls]
        x = styles[:, cols]
        fitted = x @ np.linalg.lstsq(x, dmd, rcond=None)[0]
        residual = dmd - fitted
        residual_ss = residual @ residual
        result.append(dict(
            controls=spec, score_r2=1 - residual_ss / ss,
            raw_payoff=raw, explained_payoff=(fitted @ y) / gross,
            remaining_payoff=(residual @ y) / gross,
            raw_slope=dmd @ y / ss,
            adjusted_slope=residual @ y / residual_ss if residual_ss > 1e-12 else None,
            score_correlation=(dmd @ x[:, 0]) / np.sqrt(ss * (x[:, 0] @ x[:, 0]))
            if len(cols) == 1 and np.any(x[:, 0]) else None,
        ))
    return result


def make_style_attribution(panel: pl.DataFrame) -> None:
    """Save daily DMD style data and a summary table."""
    # This calculation needs licensed sf-quant style scores.
    rows = []
    coverage = []
    panel = panel.filter((pl.col('date') >= date(2010, 1, 1)) &
                         (pl.col('label_available_date') <= date(2024, 12, 31)))
    panel = panel.select('signal_date', 'label_available_date', 'barrid',
                         'dmd', 'forward_residual_return')
    for year in range(2009, 2025):
        subset = panel.filter(pl.col('signal_date').dt.year() == year)
        if subset.is_empty():
            continue
        subset = join_style_controls(subset)
        for group in subset.partition_by('signal_date', maintain_order=True):
            day = group['signal_date'][0]
            eligible = group.filter(pl.col('dmd').is_finite() &
                                    pl.col('forward_residual_return').is_finite())
            matched = eligible.filter(pl.all_horizontal(
                pl.col(c).is_finite() for c in CONTROLS))
            coverage.append(dict(signal_date=day, eligible=eligible.height,
                                 matched=matched.height))
            if matched.height >= 30:
                # Project DMD onto each set of style scores on this date.
                rows.extend(dict(signal_date=day, stocks=matched.height, **item)
                            for item in daily_attribution(
                                matched['dmd'].to_numpy(),
                                matched['forward_residual_return'].to_numpy(),
                                matched.select(*CONTROLS).to_numpy()))
    daily = pl.DataFrame(rows).sort('signal_date', 'controls')
    daily.write_parquet(DATA / 'dmd_style_attribution_daily.parquet')
    pl.DataFrame(coverage).sort('signal_date').write_csv(
        DATA / 'dmd_style_attribution_coverage.csv')
    summary = daily.group_by('controls').agg(
        pl.len().alias('sessions'), pl.col('stocks').mean(),
        *(pl.col(c).mean() for c in ('score_r2', 'score_correlation',
            'raw_payoff', 'explained_payoff', 'remaining_payoff',
            'raw_slope', 'adjusted_slope'))
    ).with_columns((pl.col('explained_payoff') / pl.col('raw_payoff'))
                   .alias('payoff_share_explained'))
    summary.sort('controls').write_csv(DATA / 'dmd_style_attribution_summary.csv')

# With licensed sf-quant scores, use:
# import sf_quant.data as risk_source
# risk_source.env(root=str(Path.home()), database='production')
# make_style_attribution(pl.read_parquet(DATA / 'analysis_panel.parquet'))


## 1.5 Fit the candidate rules

Fit three ridge models. Give old observations less weight. Use half-lives of 63, 252, and 1,260 sessions and penalties of 18, 5, and 2.5, respectively. The penalty pulls both coefficients toward 0.00001. Use only stock returns known before the trade. The fourth rule takes the mean of the two standardized signals.

In [13]:
@dataclass(frozen=True)
class RidgeFit:
    as_of: date
    half_life_days: float
    penalty: float
    intercept: float
    coefficients: np.ndarray
    means: np.ndarray
    scales: np.ndarray
    n_observations: int

    def predict(self, signal_matrix: np.ndarray) -> np.ndarray:
        x = np.asarray(signal_matrix, dtype=float)
        if x.ndim != 2 or x.shape[1] != self.coefficients.size:
            raise ValueError(f"Expected matrix with {self.coefficients.size} signal columns")
        return self.intercept + ((x - self.means) / self.scales) @ self.coefficients

def daily_ridge_inputs(observations: pl.DataFrame, signals):
    """Compress labeled rows to daily sufficient statistics for repeated fits."""
    target = "forward_residual_return"
    columns = (*signals, target)
    products = [(a, b) for i, a in enumerate(signals) for b in columns[i:]]
    daily = (
        observations.filter(pl.all_horizontal(pl.col(c).is_finite() for c in columns))
        .group_by("signal_date", "label_available_date")
        .agg(pl.len().alias("n"), *(pl.col(c).sum() for c in columns),
             *((pl.col(a) * pl.col(b)).sum().alias(f"{a}/{b}") for a, b in products))
        .sort("signal_date")
    )
    return daily, products, signals

def fit_daily_ridge(inputs, *, as_of: date, half_life_days: float, penalty: float, center: float = 0.0) -> RidgeFit:
    """Fit trailing standardized ridge from the same moments as the row-level fit."""
    daily, products, signals = inputs
    history = daily.filter(pl.col("label_available_date") < as_of)
    n = history["n"].to_numpy()
    if n.sum() < 30:
        raise ValueError(f"Only {n.sum()} matured observations before {as_of}")
    weights = np.exp2(-np.arange(len(n) - 1, -1, -1) / half_life_days)
    total = weights @ n
    means = weights @ history.select(*signals, "forward_residual_return").to_numpy() / total
    weighted_products = weights @ history.select(*(f"{a}/{b}" for a, b in products)).to_numpy() / total
    covariance = np.zeros((len(signals), len(signals)))
    cross = np.zeros(len(signals))
    for (a, b), moment in zip(products, weighted_products):
        i = signals.index(a)
        if b in signals:
            j = signals.index(b)
            covariance[i, j] = covariance[j, i] = moment - means[i] * means[j]
        else:
            cross[i] = moment - means[i] * means[-1]
    scales = np.maximum(np.sqrt(np.maximum(covariance.diagonal(), 0)), 1e-12)
    gram = covariance / scales[:, None] / scales[None, :]
    coefficients = np.linalg.solve(gram + penalty * np.eye(len(signals)), cross / scales + penalty * center)
    return RidgeFit(as_of, half_life_days, penalty, means[-1], coefficients,
                    means[:-1], scales, int(n.sum()))

def training_inputs(output, signal):
    daily = pl.read_parquet([output / f'moments_{signal}_{y}.parquet'
                             for y in range(2005, 2025)]).sort('signal_date')
    columns = SIGNALS[signal]
    products = [(a, b) for i, a in enumerate(columns)
                for b in (*columns, 'forward_residual_return')[i:]]
    return daily, products, columns


def candidate_scores(day, history, trade_date):
    scores = day.select(history[2]).to_numpy()
    for rule, (half_life, penalty) in RULES.items():
        equal = rule == 'equal_weight'
        fit = fit_daily_ridge(history, as_of=trade_date, half_life_days=half_life,
                              penalty=penalty, center=0 if equal else 1e-5)
        z = np.nan_to_num((scores - fit.means) / fit.scales)
        yield z.mean(axis=1) if equal else z @ fit.coefficients




## 1.6 Set weights on the candidate rules

Test the four rules with the long-short constraint. Start with equal weights on the rules. At each trade, add the active returns known before the trade for each rule. Apply the softmax function to the four sums multiplied by 16. Use these model weights in both portfolio constraints. Before blending, put each candidate score on a common cross-sectional standard deviation scale. The model weights apply to forecasts.

In [14]:
def calculate_posteriors(output):
    for signal in BMA:
        history = pl.read_parquet([output / f'candidates_{year}.parquet'
                                   for year in range(2010, 2025)]).filter(pl.col('signal') == signal)
        wide = history.pivot(on='rule', index=['date', 'return_date'], values='active_return').sort('date')
        returns = wide.select(list(RULES)).to_numpy()
        if not np.isfinite(returns).all() or wide.height != 3773:
            raise ValueError(f'Incomplete candidate history for {signal}')
        known = np.searchsorted(wide['return_date'].to_numpy(), wide['date'].to_numpy(), side='left')
        cumulative = np.vstack([np.zeros(len(RULES)), returns.cumsum(axis=0)])
        weights = softmax((0.01 / 0.025**2) * cumulative[known], axis=1)
        wide.select('date').hstack(pl.DataFrame(weights, schema=list(RULES))).write_parquet(
            output / f'posterior_{signal}.parquet')




## 1.7 Construct the portfolios

The optimizer maximizes the forecast score subject to full investment, zero forecast active market beta, and a 1% predicted annual tracking-error cap. The long-only version also requires nonnegative total stock weights. The stock weights from this step give the next holding return. For turnover, compare the target weights with the prior weights after they change with stock returns.

In [15]:
"""Solve the sf-quant Barra risk problem without building a dense asset matrix."""

import clarabel
import numpy as np
from scipy import sparse


def solve(
    alpha, exposures, covariance, specific_variance, benchmark, beta,
    *, long_only, risk_cap,
):
    """Maximize alpha under an exact second-order cone tracking-error cap.

    Decision variables are active weights and their factor exposures. Keeping
    factor exposures explicit preserves sparsity in the stock risk block.
    """
    stocks, factors = exposures.shape
    size = stocks + factors
    eig, vectors = np.linalg.eigh(covariance)
    if eig.min() < -1e-10:
        raise ValueError("Barra factor covariance is not positive semidefinite")
    root = vectors * np.sqrt(np.maximum(eig, 0))
    equalities = np.vstack([np.ones(stocks), beta])
    a = [
        sparse.bmat([
            [sparse.csc_matrix(equalities), sparse.csc_matrix((2, factors))],
            [-sparse.csc_matrix(exposures.T), sparse.eye(factors)],
        ]),
        sparse.csc_matrix((1, size)),
        sparse.bmat([
            [sparse.csc_matrix((factors, stocks)), -sparse.csc_matrix(root.T)],
            [-sparse.diags(np.sqrt(specific_variance)), sparse.csc_matrix((stocks, factors))],
        ]),
    ]
    b = [np.zeros(2 + factors), np.r_[risk_cap, np.zeros(size)]]
    cones = [clarabel.ZeroConeT(2 + factors), clarabel.SecondOrderConeT(1 + size)]
    if long_only:
        a.append(sparse.hstack([-sparse.eye(stocks), sparse.csc_matrix((stocks, factors))]))
        b.append(benchmark)
        cones.append(clarabel.NonnegativeConeT(stocks))
    settings = clarabel.DefaultSettings()
    settings.verbose = False
    settings.max_threads = 1
    result = clarabel.DefaultSolver(
        sparse.csc_matrix((size, size)), np.r_[-alpha, np.zeros(factors)],
        sparse.vstack(a, format="csc"), np.concatenate(b), cones, settings,
    ).solve()
    if result.status not in (clarabel.SolverStatus.Solved, clarabel.SolverStatus.AlmostSolved):
        raise ValueError(f"Sparse Barra optimizer failed: {result.status}")
    weights = np.asarray(result.x[:stocks])
    risk = np.linalg.norm(np.r_[root.T @ (exposures.T @ weights), np.sqrt(specific_variance) * weights])
    return weights, float(risk)


In [16]:
def run_year(data_dir, output, year, phase, limit=None):
    inputs = {signal: training_inputs(output, signal) for signal in BMA}
    posterior = {}
    if phase in ('portfolios', 'exposures'):
        for signal in BMA:
            frame = pl.read_parquet(output / f'posterior_{signal}.parquet')
            posterior[signal] = dict(zip(frame['date'].to_list(), frame.select(list(RULES)).to_numpy()))
    if phase == 'exposures':
        factors = pl.read_csv(DATA / 'risk_factor_names.csv')['factor'].to_list()

        saved = pl.read_parquet(output / f'portfolios_{year}.parquet')
        saved_returns = {
            (row['date'], row['portfolio'], row['variant']): row['active_return']
            for row in saved.iter_rows(named=True)
        }
    data = pl.read_parquet(output / f'panel_{year}.parquet').filter(
        pl.col('label_available_date') <= date(2024, 12, 31))
    days = data.partition_by('date', maintain_order=True)[:limit]

    def evaluate(day):
        trade_date, return_date = day['date'][0], day['label_available_date'][0]
        ids = day['barrid'].to_numpy()
        key = str(trade_date)
        with np.load(data_dir / f'risk_{year}.npz', allow_pickle=False) as risk:
            np.testing.assert_array_equal(risk[key + '_ids'], ids)
            exposures, covariance, specific = (risk[key + '_' + k] for k in ('E', 'F', 'S'))
        benchmark, beta, ret = [day[column].to_numpy() for column in
                                ('benchmark_weight', 'predicted_beta', 'holding_return')]

        def weights(score, long_only):
            alpha = score / max(np.max(np.abs(score)), 1e-20)
            active, te = solve(alpha, exposures, covariance, specific, benchmark, beta,
                               long_only=long_only, risk_cap=0.01)
            total = benchmark + active
            if abs(total.sum() - 1) > 1e-5 or abs(active @ beta) > 1e-5 or te > 0.010001:
                raise ValueError(f'Portfolio constraint failed on {trade_date}')
            if long_only and total.min() < -1e-6:
                raise ValueError(f'Long-only constraint failed on {trade_date}')
            return total, active, te

        rows = []
        if phase == 'candidates':
            for signal in BMA:
                for rule, score in zip(RULES, candidate_scores(day, inputs[signal], trade_date)):
                    _, active, _ = weights(score, False)
                    rows.append({'date': trade_date, 'return_date': return_date,
                                 'signal': signal, 'rule': rule, 'active_return': float(active @ ret)})
            return rows, []

        forecasts = {} if phase == 'exposures' else {
            signal: np.nan_to_num(day[signal].to_numpy())
            for signal in ('dmd', 'adjusted', 'inflation_beta')
        }
        for signal, name in BMA.items():
            matrix = np.column_stack(list(candidate_scores(day, inputs[signal], trade_date)))
            sd = matrix.std(axis=0)
            unit = np.divide(matrix, sd, out=np.zeros_like(matrix), where=sd > 0)
            forecasts[name] = unit @ posterior[signal][trade_date]
        holdings = []
        for name, score in forecasts.items():
            for variant, long_only in VARIANTS.items():
                total, active, te = weights(score, long_only)
                if phase == 'exposures':
                    error = abs(active @ ret - saved_returns[trade_date, name, variant])
                    if error > 1e-8 or exposures.shape[1] != len(factors):
                        raise ValueError(f'Factor replay mismatch on {trade_date}: {error}')
                    tilt = exposures.T @ active
                    variance_share = tilt * (covariance @ tilt) / te**2
                    rows.extend({
                        'date': trade_date, 'portfolio': name, 'variant': variant,
                        'factor': factor, 'active_exposure': float(value),
                        'share_of_forecast_variance': float(share),
                    } for factor, value, share in zip(factors, tilt, variance_share))
                    continue
                rows.append({'date': trade_date, 'return_date': return_date, 'portfolio': name,
                             'variant': variant, 'active_return': float(active @ ret),
                             'portfolio_return': float(total @ ret), 'benchmark_return': float(benchmark @ ret),
                             'forecast_tracking_error': te})
                holdings.append((name, variant, ids, total, ret))
        return rows, holdings

    results, turnover = [], []
    previous = {}
    first, last = [], []
    with ThreadPoolExecutor(max_workers=int(os.environ.get('BOTQ_WORKERS', '8'))) as pool:
        for index, (rows, holdings) in enumerate(pool.map(evaluate, days)):
            results.extend(rows)
            if not holdings:
                continue
            trade, end = rows[0]['date'], rows[0]['return_date']
            last = []
            for name, variant, ids, total, ret in holdings:
                key = name, variant
                target = dict(zip(ids, total))
                gross = total * (1 + ret)
                drift = dict(zip(ids, gross / gross.sum()))
                if index == 0:
                    first.extend({'portfolio': name, 'variant': variant, 'date': trade,
                                  'barrid': stock, 'weight': weight} for stock, weight in target.items())
                elif previous[key][0] == trade:
                    old = previous[key][1]
                    value = 0.5 * sum(abs(target.get(stock, 0) - old.get(stock, 0)) for stock in target.keys() | old.keys())
                    turnover.append({'date': trade, 'portfolio': name, 'variant': variant, 'turnover': value})
                previous[key] = end, drift
                last.extend({'portfolio': name, 'variant': variant, 'date': end,
                             'barrid': stock, 'weight': weight} for stock, weight in drift.items())
    suffix = '_pilot' if limit is not None else ''
    pl.DataFrame(results).write_parquet(output / f'{phase}_{year}{suffix}.parquet')
    if phase == 'portfolios':
        pl.DataFrame(turnover).write_parquet(output / f'turnover_{year}{suffix}.parquet')
        pl.DataFrame(first).write_parquet(output / f'first_{year}{suffix}.parquet')
        pl.DataFrame(last).write_parquet(output / f'last_{year}{suffix}.parquet')




In [17]:
def collect(output):
    returns = pl.read_parquet([output / f'portfolios_{y}.parquet' for y in range(2010, 2025)])
    if returns.height != 3773 * 10 or returns.unique(['date', 'portfolio', 'variant']).height != returns.height:
        raise ValueError('Incomplete portfolio returns')
    turnover = pl.read_parquet([output / f'turnover_{y}.parquet' for y in range(2010, 2025)])
    boundaries = []
    for y in range(2011, 2025):
        old = pl.read_parquet(output / f'last_{y-1}.parquet')
        new = pl.read_parquet(output / f'first_{y}.parquet')
        if old['date'][0] != new['date'][0]:
            raise ValueError(f'Turnover date gap at year {y}')
        keys = ['portfolio', 'variant', 'date', 'barrid']
        boundaries.append(old.join(new, on=keys, how='full', coalesce=True, suffix='_new')
                          .group_by(keys[:-1]).agg(
                              (0.5 * (pl.col('weight').fill_null(0) - pl.col('weight_new').fill_null(0))
                               .abs().sum()).alias('turnover')).select(turnover.columns))
    turnover = pl.concat([turnover, *boundaries])
    if turnover.height != 3772 * 10 or turnover.unique(['date', 'portfolio', 'variant']).height != turnover.height:
        raise ValueError('Incomplete daily turnover')
    returns.sort('date', 'variant', 'portfolio').write_parquet(output / 'regular_dmd_returns.parquet')
    turnover.sort('date', 'variant', 'portfolio').write_parquet(output / 'regular_dmd_turnover.parquet')
    pl.read_parquet([output / f'coverage_{y}.parquet' for y in range(2005, 2025)]).sort('date').write_parquet(
        output / 'regular_dmd_coverage.parquet')



## 1.8 Replay the portfolio test

Set the path to your licensed score directory. Use the same score snapshot that produced the saved results to check exact agreement. The full replay can exceed 15 minutes. It writes to a separate folder. The function runs the candidate tests, computes model weights, runs both portfolio constraints, and collects the results. Leave the example calls commented to run the notebook with the supplied data.

In [18]:
def replay(score_dir: Path, output: Path, *, inputs=DATA, factor_exposures=False):
    output.mkdir(parents=True, exist_ok=True)
    for year in range(2005, 2025):
        prepare_year(inputs / 'analysis_panel.parquet', output, year, score_dir)
    for year in range(2010, 2025):
        run_year(inputs, output, year, 'candidates')
    calculate_posteriors(output)
    for year in range(2010, 2025):
        run_year(inputs, output, year, 'portfolios')
    collect(output)
    if factor_exposures:
        for year in range(2010, 2025):
            run_year(inputs, output, year, 'exposures')

def check_replay(output: Path):
    # Match rows by date and portfolio before checking numeric results.
    keys = ['date', 'portfolio', 'variant']
    for name in ('regular_dmd_returns.parquet', 'regular_dmd_turnover.parquet'):
        expected = pd.read_parquet(DATA / name).sort_values(keys).reset_index(drop=True)
        observed = pd.read_parquet(output / name).sort_values(keys).reset_index(drop=True)
        pd.testing.assert_frame_equal(observed[expected.columns], expected,
                                      rtol=1e-7, atol=1e-8)

# replay(Path('/path/to/sf-quant/scores'), Path('replay'), factor_exposures=True)
# check_replay(Path('replay'))

## 1.9 Check the supplied return and turnover data

Check that every portfolio has the expected sessions and that daily active return is the difference between total return and benchmark return.

In [19]:
returns = pd.read_parquet(DATA / 'regular_dmd_returns.parquet')
turnover = pd.read_parquet(DATA / 'regular_dmd_turnover.parquet')
keys = ['date', 'portfolio', 'variant']
for frame, sessions in [(returns, 3773), (turnover, 3772)]:
    assert frame.groupby(keys[1:]).size().eq(sessions).all()
    assert frame.groupby(keys[1:]).ngroups == 10
    assert not frame.duplicated(keys).any()
np.testing.assert_allclose(returns['active_return'],
                           returns['portfolio_return'] - returns['benchmark_return'],
                           atol=1e-12)
assert returns['forecast_tracking_error'].between(0, 0.010001).all()
assert turnover['turnover'].ge(0).all()
assert (returns['date'] < returns['return_date']).all()
assert returns.groupby('return_date')['benchmark_return'].nunique().eq(1).all()
assert returns['return_date'].nunique() == 3773
assert set(map(tuple, turnover[keys].to_numpy())) <= set(map(tuple, returns[keys].to_numpy()))
print('Checked 10 return series and 10 turnover series.')

Checked 10 return series and 10 turnover series.
